# 04 · Real public data: profile → prepare → evaluate (NYC 311 · Chicago 311 · RDD2022)

> **REAL PUBLIC DATA workflow.** Real datasets are acquired *outside* git, under their own licences. Nothing downloaded here is committed or redistributed; the output bundle contains **aggregate statistics only** (no records, no images). Read each dataset's terms yourself: the source cards in `ai/training/src/data_sources/cards/` record what is and is not verified (currently NONE of the licences is verified at the primary source). Real-data results must always be reported with their provenance and never mixed into a single accuracy figure with synthetic results.

**Settings:** CPU; **Internet On** only if the notebook downloads (Socrata). For RDD2022, attach an existing copy as a Kaggle Dataset input and set `INPUT_PATH` (the archive is multi-GB; verify the copy's provenance/licence too). Choose `SOURCE` below. Nothing runs until `ACCEPT_TERMS = True`.

In [ ]:
# --- 0. Locate the CivicConnect repo and set up a work directory (works on Kaggle and locally) ---------------
import importlib.util
import os
import shutil
import subprocess
import sys
import tempfile
from pathlib import Path

ON_KAGGLE = bool(os.environ.get("KAGGLE_KERNEL_RUN_TYPE"))

def find_repo() -> Path:
    """Return a writable directory that contains ai/inference/schemas.py."""
    marker = Path("ai/inference/schemas.py")
    if os.environ.get("CIVIC_REPO"):                                   # explicit override
        return Path(os.environ["CIVIC_REPO"])
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:                                    # running from inside the repo
        if (p / marker).exists():
            return p
    if ON_KAGGLE:                                                      # attached Kaggle Dataset containing the repo or just ai/
        for hit in sorted(Path("/kaggle/input").glob("**/ai/inference/schemas.py"))[:1]:
            src = hit.parents[2]                                       # .../ai/inference/schemas.py -> parent of ai/
            dst = Path("/kaggle/working/repo")
            if not dst.exists():
                shutil.copytree(src, dst)                              # /kaggle/input is read-only
            return dst
        url = os.environ.get("CIVIC_GIT_URL", "https://github.com/tanujb03/CivicConnect_v2")                          # default: clone the project repo's `tanuj` branch (needs Internet ON); override with CIVIC_GIT_URL / CIVIC_GIT_REF
        if url:
            subprocess.run(["git", "clone", "--depth", "1", "--branch", os.environ.get("CIVIC_GIT_REF", "tanuj"), url, "/kaggle/working/repo"], check=True)
            return Path("/kaggle/working/repo")
    raise SystemExit("Cannot find the repo. On Kaggle: turn Internet ON (the notebook clones the project from GitHub) or add a Dataset containing the repo's `ai/` folder (see ai/training/README.md).")

REPO = find_repo()
sys.path.insert(0, str(REPO))
WORK = Path(os.environ.get("CIVIC_WORKDIR") or ("/kaggle/working" if ON_KAGGLE else tempfile.mkdtemp(prefix="civic_work_")))
WORK.mkdir(parents=True, exist_ok=True)

_MODULE = {"scikit-learn": "sklearn", "sentence-transformers": "sentence_transformers"}

def ensure(*pkgs):
    """pip install only what is missing (Kaggle images already ship numpy/scipy/scikit-learn/pydantic/httpx)."""
    missing = [p for p in pkgs if importlib.util.find_spec(_MODULE.get(p, p)) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

ensure("numpy", "scipy", "scikit-learn", "pydantic", "httpx")
print(f"repo : {REPO}\nwork : {WORK}\npython {sys.version.split()[0]} | kaggle={ON_KAGGLE}")

In [ ]:
# ---- parameters (edit these) ----
SOURCE = os.environ.get("CIVIC_REAL_SOURCE", "chicago311")        # "nyc311" | "chicago311" | "rdd2022"
ACCEPT_TERMS = False                                              # set True ONLY after reading the printed terms and the dataset's own terms page
ACK_UNVERIFIED_LICENSE = False                                    # set True to proceed locally while the card's licence is still UNVERIFIED (results must not be published)
INPUT_PATH = os.environ.get("CIVIC_REAL_INPUT")                   # attached Kaggle dataset path; if None the 311 sources are downloaded via the Socrata API
SINCE, UNTIL, MAX_ROWS = "2024-01-01", None, 20000                # bounded slice (311 sources)
HOLDOUT_AFTER = "2025-06-01"                                      # 311: most recent requests form the holdout; RDD: use HOLDOUT_COUNTRIES
HOLDOUT_COUNTRIES = ["India"]
IMAGE_ROOT = INPUT_PATH                                           # RDD2022 root (images are read in place, never copied)

# CI hook ONLY: run on the repo's invented format fixtures (NOT real data) so this notebook can be tested without network access
if os.environ.get("CIVIC_REAL_FIXTURE") == "1":
    ACCEPT_TERMS = ACK_UNVERIFIED_LICENSE = True
    SOURCE = "chicago311"
    INPUT_PATH = str(REPO / "ai/training/tests/fixtures/real_data_formats/chicago311_format_sample.jsonl")
    HOLDOUT_AFTER = "2025-09-04"

In [ ]:
from ai.training.src.data_sources import cli
from ai.training.src.data_sources.card_schema import load_card

card = load_card(SOURCE)
print(card.terms_summary())
print("\nSupports :", [(s.capability, s.how[:90]) for s in card.supports])
print("Does NOT support:", [g.capability for g in card.does_not_support])
if not ACCEPT_TERMS:
    raise SystemExit("Stopped: review the terms above, then set ACCEPT_TERMS = True (and ACK_UNVERIFIED_LICENSE = True while the licence is unverified).")
TERMS = ["--accept-terms", SOURCE] + (["--acknowledge-unverified-license"] if ACK_UNVERIFIED_LICENSE else [])
REAL = WORK / "real" / SOURCE            # OUTSIDE the git repo (the CLI refuses repo paths)
REAL.mkdir(parents=True, exist_ok=True)

## 1 · Acquire
311 sources: bounded Socrata pull (Internet On). RDD2022: use an attached copy (`INPUT_PATH`).

In [ ]:
if INPUT_PATH is None:
    if SOURCE == "rdd2022":
        raise SystemExit("RDD2022 is multi-GB: attach it as a Kaggle Dataset and set INPUT_PATH. To check the licence at the primary source (Internet On): "
                         "python -m ai.training.src.data_sources.cli figshare-license")
    args = ["download", SOURCE, "--out", str(REAL / "raw"), "--max-rows", str(MAX_ROWS), "--since", SINCE] + (["--until", UNTIL] if UNTIL else []) + TERMS
    assert cli.main(args) == 0
    INPUT_PATH = str(REAL / "raw" / "raw.jsonl")
print("input:", INPUT_PATH)

## 2 · Profile (tabular sources) — verify columns, the real value space, free-text columns and the draft mapping coverage BEFORE trusting any mapping

In [ ]:
import json

if SOURCE != "rdd2022":
    assert cli.main(["profile", SOURCE, "--input", INPUT_PATH, "--out", str(REAL / "profile.json")]) == 0
    prof = json.loads((REAL / "profile.json").read_text(encoding="utf-8"))
    print("unresolved column aliases :", prof["unresolved_aliases"])
    print("coordinate coverage       :", prof["coordinate_coverage"], "| date range:", prof["date_range"])
    print("free-text column candidates:", prof["candidate_free_text_columns"] or "none -> no narrative text available for text-intake evaluation")
    print("top source labels         :", prof["top_values"][list(prof["top_values"])[0]][:10])

## 3 · Prepare into the canonical schema (explicit, versioned, DRAFT taxonomy mapping)

In [ ]:
prep_args = ["prepare", SOURCE, "--input", INPUT_PATH, "--out", str(REAL / "prepared")] + TERMS
if SOURCE == "rdd2022":
    prep_args += ["--holdout-countries", *HOLDOUT_COUNTRIES]
else:
    prep_args += ["--holdout-after", HOLDOUT_AFTER, "--max-rows", str(MAX_ROWS)]
assert cli.main(prep_args) == 0
man = json.loads((REAL / "prepared" / "PREPARE_MANIFEST.json").read_text(encoding="utf-8"))
print("licence verified:", man["license_verified"], "| records:", man["records"], "| splits:", man["split_counts"])
print("mapping coverage:", (man.get("coverage") or man["box_mapping_coverage"])["share"])

## 4 · Duplicate pairs from agency links (Chicago only) and real-data evaluation tracks

In [ ]:
from ai.evaluation import run_eval

REPORTS = WORK / "reports_real"
REC = str(REAL / "prepared" / "records.jsonl")
if SOURCE != "rdd2022":
    assert run_eval.main(["--task", "taxonomy_coverage", "--real-data", REC, "--out", str(REPORTS), "--name", f"{SOURCE}_taxonomy_coverage"]) == 0
if SOURCE == "chicago311":
    assert cli.main(["pairs", "--records", REC, "--out", str(REAL / "pairs")]) == 0
    pairs_man = json.loads((REAL / "pairs" / "PAIRS_MANIFEST.json").read_text(encoding="utf-8"))
    if pairs_man["stats"]["positives"]:
        assert run_eval.main(["--task", "fusion", "--track", "real_holdout", "--real-data", str(REAL / "pairs" / "pairs.jsonl"),
                              "--out", str(REPORTS), "--name", "chicago311_fusion_real_holdout"]) == 0
    else:
        print("No agency-linked duplicates found: the file lacks duplicate/parent columns or they are empty (see the profile).")
if SOURCE == "rdd2022":
    print("Vision evaluation needs a provider (credentials + validated model ID):\n"
          "  python -m ai.evaluation.run_eval --task vision --track real_holdout --real-data <prepared/records.jsonl> --image-root <RDD root> --system provider")

## 5 · Bundle AGGREGATES only (reports + manifests). Records/images are deliberately excluded.

In [ ]:
import hashlib
import zipfile

BUNDLE = WORK / f"civic_real_{SOURCE}_aggregates.zip"
with zipfile.ZipFile(BUNDLE, "w", zipfile.ZIP_DEFLATED) as z:
    for f in [REAL / "profile.json", REAL / "prepared" / "PREPARE_MANIFEST.json", REAL / "pairs" / "PAIRS_MANIFEST.json", REAL / "raw" / "RETRIEVAL.json"]:
        if f.exists():
            z.write(f, f"{SOURCE}/{f.name}")
    for f in sorted(REPORTS.glob("*")) if REPORTS.exists() else []:
        z.write(f, f"reports/{f.name}")
    assert not any(n.endswith(("records.jsonl", "pairs.jsonl", "raw.jsonl", ".jpg")) for n in z.namelist())
print("bundle (aggregates only):", BUNDLE, "|", BUNDLE.stat().st_size, "bytes | sha256", hashlib.sha256(BUNDLE.read_bytes()).hexdigest()[:16])
print("REMINDER: do not commit or redistribute the data; keep the licence status in every report.")